In [15]:
# 0. IMPORTS, PATHS, FOLDER STRUCTURE
# ============================================================================
# Step 0 of the pipeline: detect tracks that are not in any playlist or liked
# songs. Save the list of orphan URIs for manual Spotify upload.
#
# Runs BEFORE 00_1_build_full_history.ipynb.
# ============================================================================

import sys
import glob
import json
from pathlib import Path

import numpy as np
import pandas as pd


def find_src(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / 'src').is_dir() and (p / '01_data_analysis').is_dir():
            return p / 'src'
    raise FileNotFoundError('src/ not found above current directory')


SRC_PATH = find_src()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from paths import get_data_path, get_project_root

# ---------- Base paths ----------
PROJECT_ROOT = get_project_root()
DATA = get_data_path()

# ---------- Full structure ----------
PATHS = {
    'streaming_history': DATA / 'Spotify' / 'Spotify_ Extended_Streaming_History',
    'account_data':      DATA / 'Spotify' / 'Spotify_Account_ Data',
    'exportify':         DATA / 'Exportify',
    'orphan_export':     DATA / 'Exportify' / 'exportify_orphan_export',
    'orphans_to_add':    DATA / 'Exportify' / 'orphans_to_add',
    'output':            DATA / '00_output',
}

# ---------- Create folders if missing ----------
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# ---------- Glob patterns: nested + flat fallback ----------
def _find(nested_pattern, flat_pattern):
    files = sorted(glob.glob(nested_pattern))
    if not files:
        files = sorted(glob.glob(flat_pattern))
    return files

history_files = _find(
    str(PATHS['streaming_history'] / '*' / 'Streaming_History_Audio_*.json'),
    str(PATHS['streaming_history'] / 'Streaming_History_Audio_*.json'),
)

exportify_files = _find(
    str(PATHS['exportify'] / '*' / 'spotify_playlists (*)' / '*.csv'),
    str(PATHS['exportify'] / 'spotify_playlists (*)' / '*.csv'),
)

orphan_files = sorted(glob.glob(str(PATHS['orphan_export'] / '*.csv')))

# Account Data JSON counter
json_nested = sorted(glob.glob(str(PATHS['account_data'] / '*' / '*.json')))
json_flat = sorted(glob.glob(str(PATHS['account_data'] / '*.json')))
account_json_files = json_nested if json_nested else json_flat

# ---------- Constants ----------
BATCH_SIZE = 10_000

# ---------- Status ----------
print('=' * 78)
print('DATA STRUCTURE')
print('=' * 78)
print()
print(f'  Streaming History      : {len(history_files)} files')
print(f'  Account Data           : {len(account_json_files)} JSON files')
print(f'  Exportify playlists    : {len(exportify_files)} CSV files')
print(f'  Orphan export          : {len(orphan_files)} CSV files')
print()
print('=' * 78)
print('WHAT TO PUT WHERE')
print('=' * 78)
print()
print('  1. data/Spotify/Spotify_ Extended_Streaming_History/')
print('       Streaming_History_Audio_*.json')
print()
print('  2. data/Spotify/Spotify_Account_ Data/')
print('       YourLibrary.json, Playlist*.json, UserAttributes.json, ...')
print()
print('  3. data/Exportify/spotify_playlists (<account>)/')
print('       <playlist>.csv, Liked_Songs.csv')
print()
print('  4. data/Exportify/exportify_orphan_export/')
print('       <orphan>.csv (filled later, after Exportify)')
print()
print('  5. data/Exportify/orphans_to_add/')
print('       orphans_uris.csv, orphans_batch_NN.csv (created by this notebook)')
print()
print('  6. data/00_output/')
print('       intermediate parquet files (created by 00_1 / 00_2)')
print()

if len(history_files) == 0:
    print('WARNING: No history JSON found.')
if len(exportify_files) == 0:
    print('WARNING: No Exportify playlist CSV found.')
if len(history_files) > 0 and len(exportify_files) > 0:
    print('Ready. Run the next cell.')

DATA STRUCTURE

  Streaming History      : 15 files
  Account Data           : 38 JSON files
  Exportify playlists    : 364 CSV files
  Orphan export          : 8 CSV files

WHAT TO PUT WHERE

  1. data/Spotify/Spotify_ Extended_Streaming_History/
       Streaming_History_Audio_*.json

  2. data/Spotify/Spotify_Account_ Data/
       YourLibrary.json, Playlist*.json, UserAttributes.json, ...

  3. data/Exportify/spotify_playlists (<account>)/
       <playlist>.csv, Liked_Songs.csv

  4. data/Exportify/exportify_orphan_export/
       <orphan>.csv (filled later, after Exportify)

  5. data/Exportify/orphans_to_add/
       orphans_uris.csv, orphans_batch_NN.csv (created by this notebook)

  6. data/00_output/
       intermediate parquet files (created by 00_1 / 00_2)

Ready. Run the next cell.


In [16]:
# 0.5. SPOTIFY API CREDENTIALS
# ============================================================================
# Loads client_id / client_secret / redirect_uri from a local JSON file.
# The file is ignored by git and never committed.
# ============================================================================

CREDS_PATH = Path.cwd() / 'spotify_credentials.json'
if not CREDS_PATH.exists():
    CREDS_PATH = Path(SRC_PATH).parent / '01_data_analysis' / '00_prep' / 'spotify_credentials.json'

if not CREDS_PATH.exists():
    print('WARNING: spotify_credentials.json not found.')
    print('   Create the file next to the notebook.')
    SPOTIFY = None
else:
    with open(CREDS_PATH, 'r', encoding='utf-8') as f:
        SPOTIFY = json.load(f)

    missing = [k for k in ('client_id', 'client_secret', 'redirect_uri') if not SPOTIFY.get(k)]
    if missing:
        print(f'WARNING: missing keys in credentials: {missing}')
        SPOTIFY = None
    else:
        print('Spotify credentials loaded.')

Spotify credentials loaded.


In [17]:
# 1. UNIQUE TRACKS FROM LISTENING HISTORY
# ============================================================================

frames = []
for f in history_files:
    df = pd.read_json(f)
    df['account'] = Path(f).parent.name
    frames.append(df)

df_hist = pd.concat(frames, ignore_index=True)

df_unique = df_hist.drop_duplicates(subset=['spotify_track_uri'])[
    ['spotify_track_uri', 'master_metadata_track_name', 'master_metadata_album_artist_name']
].rename(columns={
    'master_metadata_track_name': 'track_name',
    'master_metadata_album_artist_name': 'artist_name',
})

df_unique = df_unique.dropna(subset=['spotify_track_uri'])
df_unique['spotify_track_uri'] = df_unique['spotify_track_uri'].astype(str)
df_unique = df_unique[df_unique['spotify_track_uri'].str.strip() != '']

history_uris = set(df_unique['spotify_track_uri'])

print(f'Raw events:          {len(df_hist):,}')
print(f'Unique tracks:       {len(df_unique):,}')

Raw events:          80,055
Unique tracks:       25,629


In [18]:
# 2. UNIQUE TRACKS FROM EXPORTIFY PLAYLISTS
# ============================================================================

playlist_uris = set()
n_files = 0

for f in exportify_files:
    try:
        df = pd.read_csv(f, usecols=['Track URI'])
        playlist_uris.update(df['Track URI'].dropna().astype(str))
        n_files += 1
    except Exception:
        pass

playlist_uris = {u for u in playlist_uris if u.strip() != ''}

print(f'CSV files:           {n_files}')
print(f'Unique tracks:       {len(playlist_uris):,}')

CSV files:           364
Unique tracks:       24,953


In [19]:
# 3. COMPARISON
# ============================================================================
# Three groups:
#   1. In history AND in playlists   - listened, saved to playlists
#   2. Only in history               - ORPHANS
#   3. Only in playlists             - saved but never listened
# ============================================================================

in_both   = history_uris & playlist_uris
only_hist = history_uris - playlist_uris     # ORPHANS
only_pl   = playlist_uris - history_uris

print('=' * 70)
print('COMPARISON')
print('=' * 70)
print()
print(f'  History:                     {len(history_uris):,}')
print(f'  Playlists:                   {len(playlist_uris):,}')
print()
print(f'  1. In history AND playlists: {len(in_both):,}')
print(f'  2. Only in history (ORPHANS):{len(only_hist):,}')
print(f'  3. Only in playlists:        {len(only_pl):,}')
print()
print(f'  Check: {len(in_both):,} + {len(only_hist):,} = {len(in_both) + len(only_hist):,}')

COMPARISON

  History:                     25,629
  Playlists:                   24,953

  1. In history AND playlists: 7,234
  2. Only in history (ORPHANS):18,395
  3. Only in playlists:        17,719

  Check: 7,234 + 18,395 = 25,629


In [20]:
# 4. SAVE ORPHAN LISTS
# ============================================================================
# Saves:
#   orphans_uris.csv       - all orphans in one file
#   orphans_batch_NN.csv   - batches of 10,000 (Spotify playlist limit)
# ============================================================================

only_hist = {u for u in only_hist if u and str(u).strip() != ''}

ORPHANS_OUT = PATHS['orphans_to_add']
orphans_sorted = sorted(only_hist)

pd.DataFrame({'spotify_track_uri': orphans_sorted}).to_csv(
    ORPHANS_OUT / 'orphans_uris.csv', index=False
)
print(f'  orphans_uris.csv       : {len(orphans_sorted):,}')

n_batches = (len(orphans_sorted) + BATCH_SIZE - 1) // BATCH_SIZE

for i in range(n_batches):
    chunk = orphans_sorted[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
    pd.DataFrame({'spotify_track_uri': chunk}).to_csv(
        ORPHANS_OUT / f'orphans_batch_{i+1:02d}.csv', index=False
    )
    print(f'  orphans_batch_{i+1:02d}.csv : {len(chunk):,}')

print()
print(f'  Total orphans:   {len(orphans_sorted):,}')
print(f'  Batches:         {n_batches}')
print(f'  Output folder:   data/Exportify/orphans_to_add/')

  orphans_uris.csv       : 18,395
  orphans_batch_01.csv : 10,000
  orphans_batch_02.csv : 8,395

  Total orphans:   18,395
  Batches:         2
  Output folder:   data/Exportify/orphans_to_add/


In [21]:
# 5. BRANCH: 'api' OR 'manual'
# ============================================================================
# METHOD = 'manual'
METHOD = 'api'   # 'api' or 'manual'

In [22]:
# 6A. CREATE ORPHAN PLAYLISTS VIA SPOTIFY API
# ============================================================================
# Requires:
#   - spotify_credentials.json next to the notebook
#   - Spotify Premium account
#   - OAuth (browser opens once)
#
# Creates N private playlists: __ORPHANS_01, __ORPHANS_02, ...
# Adds tracks in batches of 100 (API limit).
# Saves metadata to orphans_to_add/created_playlists.json
# ============================================================================

if METHOD == 'api':

    if SPOTIFY is None:
        print('No Spotify credentials. Check spotify_credentials.json and cell 0.5.')

    else:
        import spotipy
        from spotipy.oauth2 import SpotifyOAuth

        # ---------- OAuth ----------
        sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
            client_id=SPOTIFY['client_id'],
            client_secret=SPOTIFY['client_secret'],
            redirect_uri=SPOTIFY['redirect_uri'],
            scope='playlist-modify-private playlist-read-private',
            cache_path='.spotify_cache',
        ))

        me = sp.me()
        print('Spotify authorization OK.')
        print()

        # ---------- Existing playlists (pagination) ----------
        existing = {}
        offset = 0
        while True:
            page = sp._get('me/playlists', limit=50, offset=offset)
            for pl in page['items']:
                if pl and pl.get('name'):
                    existing[pl['name']] = pl
            if page.get('next') is None:
                break
            offset += 50

        # ---------- Duplicate protection ----------
        created = []
        skipped = []
        conflict = []

        for i in range(n_batches):
            chunk = orphans_sorted[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
            pl_name = f'__ORPHANS_{i+1:02d}'

            if pl_name in existing:
                pl = existing[pl_name]
                info = sp._get(f'playlists/{pl["id"]}/items', limit=1)
                n_existing = info['total']

                if n_existing == len(chunk):
                    print(f'  {pl_name}: already exists ({n_existing:,}) - skipping')
                    skipped.append({
                        'name': pl_name,
                        'id': pl['id'],
                        'url': pl['external_urls']['spotify'],
                        'n_tracks': n_existing,
                        'status': 'skipped',
                    })
                    continue
                else:
                    print(f'  {pl_name}: exists with {n_existing:,}, expected {len(chunk):,}')
                    conflict.append({
                        'name': pl_name,
                        'existing_id': pl['id'],
                        'existing_n': n_existing,
                        'expected_n': len(chunk),
                    })
                    continue

            # ---------- Create ----------
            print(f'  {pl_name}: {len(chunk):,} tracks')
            pl = sp._post('me/playlists', payload={
                'name': pl_name,
                'public': False,
                'description': f'Orphan tracks batch {i+1}/{n_batches}',
            })
            pid = pl['id']

            for j in range(0, len(chunk), 100):
                sp._post(f'playlists/{pid}/items', payload={'uris': chunk[j : j + 100]})

            print(f'     added {len(chunk):,}')
            created.append({
                'name': pl_name,
                'id': pid,
                'url': pl['external_urls']['spotify'],
                'n_tracks': len(chunk),
                'status': 'created',
            })

        # ---------- Conflicts ----------
        if conflict:
            print()
            print('=' * 70)
            print('CONFLICTS (same name, different track count)')
            print('=' * 70)
            for c in conflict:
                print(f"  {c['name']}: in Spotify {c['existing_n']:,}, expected {c['expected_n']:,}")
            print()
            print('  Options:')
            print('    A. Delete these playlists in Spotify manually, then re-run.')
            print('    B. Rename them in Spotify.')
            print('    C. Leave as is - these batches will not be created.')
            print()

        # ---------- Metadata ----------
        meta_path = ORPHANS_OUT / 'created_playlists.json'
        meta = created + skipped
        with open(meta_path, 'w', encoding='utf-8') as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

        # ---------- Summary ----------
        print()
        print('=' * 70)
        print(f'  Created:    {len(created)}')
        print(f'  Skipped:    {len(skipped)} (already existed)')
        print(f'  Conflicts:  {len(conflict)}')
        print(f'  Metadata:   data/Exportify/orphans_to_add/created_playlists.json')
        print('=' * 70)
        print()
        print('NEXT STEPS:')
        print('  1. Open https://exportify.net/ and authorize with Spotify')
        print('  2. Find playlists __ORPHANS_01, __ORPHANS_02, ...')
        print('  3. Export each - downloads CSV with audio features')
        print('  4. Put CSVs into data/Exportify/exportify_orphan_export/')
        print('  5. Run 00_1_build_full_history.ipynb')

Spotify authorization OK.

  __ORPHANS_01: already exists (10,000) - skipping
  __ORPHANS_02: already exists (8,395) - skipping

  Created:    0
  Skipped:    2 (already existed)
  Conflicts:  0
  Metadata:   data/Exportify/orphans_to_add/created_playlists.json

NEXT STEPS:
  1. Open https://exportify.net/ and authorize with Spotify
  2. Find playlists __ORPHANS_01, __ORPHANS_02, ...
  3. Export each - downloads CSV with audio features
  4. Put CSVs into data/Exportify/exportify_orphan_export/
  5. Run 00_1_build_full_history.ipynb


In [25]:
# 6b. MANUAL PLAYLIST CREATION
# ============================================================================

if METHOD == 'manual':

    print('=' * 78)
    print('MANUAL ORPHAN PLAYLIST CREATION')
    print('=' * 78)
    print()
    print(f'  Orphans total:    {len(orphans_sorted):,}')
    print(f'  Batch files:      {n_batches}')
    print(f'  Location:         data/Exportify/orphans_to_add/')
    print()
    print('=' * 78)
    print('STEP 1. CREATE PRIVATE PLAYLISTS IN SPOTIFY')
    print('=' * 78)
    print()
    print('  Open Spotify DESKTOP APP.')
    print('  WARNING: the web version does NOT support bulk URI paste.')
    print()
    print('  Create one playlist per batch file. Names (exact):')
    print()
    for i in range(n_batches):
        n_in_batch = min(BATCH_SIZE, len(orphans_sorted) - i * BATCH_SIZE)
        print(f'      __ORPHANS_{i+1:02d}      ({n_in_batch:,} tracks)')
    print()
    print('  IMPORTANT:')
    print('    - Set each playlist to PRIVATE (three dots -> Make private)')
    print('    - Otherwise orphans appear in your public profile')
    print()
    print('=' * 78)
    print('STEP 2. ADD TRACKS')
    print('=' * 78)
    print()
    print('  Work in the DESKTOP app, not in the browser.')
    print()
    print('  For each file orphans_batch_NN.csv:')
    print('    1. Open the CSV in Excel or a text editor')
    print('    2. Copy the entire spotify_track_uri column')
    print('    3. Open the matching playlist in Spotify')
    print('    4. Inside the playlist press Ctrl+V')
    print('       (or three dots -> Add to this playlist -> paste)')
    print('    5. Confirm')
    print()
    print('  Spotify accepts the whole batch in a single paste.')
    print('  Repeat for all batches.')
    print()
    print('  Check: playlist track count = number of rows in the CSV.')
    print()
    print('=' * 78)
    print('STEP 3. EXPORT VIA EXPORTIFY')
    print('=' * 78)
    print()
    print('  1. Open https://exportify.net/')
    print('  2. Log in with Spotify (one time)')
    print('  3. Find playlists __ORPHANS_01, __ORPHANS_02, ... in the list')
    print('  4. Click Export on each -> a CSV with audio features downloads')
    print()
    print('=' * 78)
    print('STEP 4. PUT CSV FILES IN PLACE')
    print('=' * 78)
    print()
    print('  Move all downloaded CSVs into:')
    print('      data/Exportify/exportify_orphan_export/')
    print()
    print('  Expected: same number of files as batch files.')
    print()
    print('=' * 78)
    print('STEP 5. RUN THE NEXT NOTEBOOK')
    print('=' * 78)
    print()
    print('  Run 00_1_build_full_history.ipynb.')
    print('  It will pick up the CSV files as orphan references')
    print('  and enrich the history with audio features.')
    print()
    print('=' * 78)

In [24]:
# 7. FINAL CHECK
# ============================================================================
# Shows:
#   1. How many orphans were detected in this run
#   2. How many of them were already exported (from exportify_orphan_export/)
#   3. How many exported tracks have audio features
# ============================================================================

print('=' * 70)
print('FINAL CHECK')
print('=' * 70)
print()

# ---------- Orphans from this run ----------
orphans_file = ORPHANS_OUT / 'orphans_uris.csv'
if orphans_file.exists():
    df_orphans = pd.read_csv(orphans_file)
    orphans_uris = set(df_orphans['spotify_track_uri'].astype(str))
    print(f'  Orphans (this run)         : {len(orphans_uris):,}')
else:
    orphans_uris = set()
    print('  Orphans (this run)         : 0 (file not created)')

# ---------- Already exported ----------
orphan_export_dir = PATHS['orphan_export']
exported_uris = set()
all_exported = []

if orphan_export_dir.exists():
    for f in sorted(orphan_export_dir.glob('*.csv')):
        try:
            df = pd.read_csv(f)
            if 'Track URI' in df.columns:
                exported_uris.update(df['Track URI'].dropna().astype(str))
                all_exported.append(df)
        except Exception:
            pass

exported_uris = {u for u in exported_uris if u.strip() != ''}
print(f'  Exported via Exportify     : {len(exported_uris):,}')
print()

# ---------- Coverage ----------
if len(orphans_uris) > 0 and len(exported_uris) > 0:
    covered = orphans_uris & exported_uris
    not_covered = orphans_uris - exported_uris
    print('=' * 70)
    print('COVERAGE')
    print('=' * 70)
    print(f'  Total orphans              : {len(orphans_uris):,}')
    print(f'  Covered by export          : {len(covered):,} ({len(covered)/len(orphans_uris)*100:.1f}%)')
    print(f'  Still to export            : {len(not_covered):,}')
    print()
    if len(not_covered) == 0:
        print('  All orphans exported. Ready to run 00_1.')
    else:
        print('  Some orphans are not exported yet.')

elif len(orphans_uris) > 0 and len(exported_uris) == 0:
    print('=' * 70)
    print('COVERAGE')
    print('=' * 70)
    print(f'  Orphans found              : {len(orphans_uris):,}')
    print(f'  But exportify_orphan_export/ is empty.')
    print()
    print('  Steps:')
    print('    1. Upload orphans to Spotify (API or manual)')
    print('    2. Export them via Exportify')
    print('    3. Put CSVs into data/Exportify/exportify_orphan_export/')
    print('    4. Run 00_1_build_full_history.ipynb')

elif len(orphans_uris) == 0:
    print('  No orphans detected. Check cells 1-4.')

print()
print('=' * 70)

# ---------- Audio features ----------
if all_exported:
    print()
    print('=' * 70)
    print('AUDIO FEATURES IN EXPORT')
    print('=' * 70)

    df_exported = pd.concat(all_exported, ignore_index=True)
    df_unique_exp = df_exported.drop_duplicates(subset=['Track URI']).copy()

    valence_col = None
    for c in ['Valence', 'valence']:
        if c in df_unique_exp.columns:
            valence_col = c
            break

    if valence_col is None:
        print('  Valence column not found in CSV.')
        print(f'  Columns: {df_unique_exp.columns.tolist()}')
    else:
        with_features = df_unique_exp[valence_col].notna().sum()
        without_features = df_unique_exp[valence_col].isna().sum()
        total = len(df_unique_exp)
        pct = with_features / total * 100 if total > 0 else 0

        print(f'  Tracks in export           : {total:,}')
        print(f'  With features (valence)    : {with_features:,} ({pct:.1f}%)')
        print(f'  Without features           : {without_features:,}')
        print()

        if pct >= 99:
            print('  All orphans have features.')
        elif pct >= 80:
            print(f'  Most have features ({pct:.1f}%).')
        elif pct >= 50:
            print(f'  Only {pct:.1f}% have features. Acceptable but not ideal.')
        else:
            print(f'  Low feature coverage ({pct:.1f}%). Check export.')

    print()
    print('=' * 70)

FINAL CHECK

  Orphans (this run)         : 18,395
  Exported via Exportify     : 18,742

COVERAGE
  Total orphans              : 18,395
  Covered by export          : 18,395 (100.0%)
  Still to export            : 0

  All orphans exported. Ready to run 00_1.


AUDIO FEATURES IN EXPORT
  Tracks in export           : 18,742
  With features (valence)    : 18,735 (100.0%)
  Without features           : 7

  All orphans have features.



## Содержание `00_0_orphans_detect.ipynb`

**Цель ноутбука:**
Найти треки, которые слушались, но которых нет ни в одном плейлисте и ни в лайках. Сохранить список URI для загрузки в Spotify.

**Общие принципы:**
- Пути через `src.paths`, без хардкода.
- Структура папок создаётся автоматически.
- Glob-паттерны ищут и в корне, и в подпапках.
- Вывод — счётчики и относительные пути, без абсолютных путей и реальных имён.
- Spotify-ключи читаются из `spotify_credentials.json` (в `.gitignore`).

---

### Ячейка 0. Импорты, пути, структура папок

**Что делает:**
- Импортирует библиотеки, подключает `src/` через `find_src`.
- Загружает `get_data_path`, `get_project_root`.
- Определяет словарь `PATHS` со всеми рабочими папками.
- Создаёт папки, если их нет.
- Ищет файлы через `_find()`: сначала в подпапках, если пусто — в корне.
- Считает файлы: история, Account Data, Exportify, сироты.
- Задаёт `BATCH_SIZE = 10_000`.
- Печатает статус и куда что класть.

**Выход:** `PATHS`, `history_files`, `exportify_files`, `orphan_files`, `account_json_files`, `BATCH_SIZE`.

---

### Ячейка 0.5. Spotify API credentials

**Что делает:**
- Ищет `spotify_credentials.json` рядом с ноутом или в `01_data_analysis/00_prep/`.
- Загружает JSON в переменную `SPOTIFY`.
- Проверяет наличие `client_id`, `client_secret`, `redirect_uri`.
- Если чего-то нет — `SPOTIFY = None`, печатает предупреждение.

**Выход:** `SPOTIFY` (dict или None).

**Примечание:** файл в `.gitignore`, ключи не уходят в git.

---

### Ячейка 1. Уникальные треки из истории

**Вход:** все `Streaming_History_Audio_*.json` в `data/Spotify/Spotify_ Extended_Streaming_History/`.
**Выход:** `df_hist`, `df_unique`, `history_uris`.

**Логика:**
- Читает все JSON, добавляет `account` (имя папки).
- Конкатенирует в `df_hist`.
- Дедуплицирует по `spotify_track_uri`, оставляет первую встречу.
- Переименовывает `master_metadata_track_name` → `track_name`, `master_metadata_album_artist_name` → `artist_name`.
- Убирает строки с пустым URI.
- Собирает множество `history_uris`.

**Печатает:** сырых событий, уникальных треков.

---

### Ячейка 2. Уникальные треки из Exportify

**Вход:** все CSV в `data/Exportify/spotify_playlists (*)/`.
**Выход:** `playlist_uris`.

**Логика:**
- Читает каждый CSV, берёт колонку `Track URI`.
- Обновляет глобальный `set()`. Дедупликация общая, не по аккаунтам.
- Убирает пустые строки.

**Печатает:** количество CSV, уникальных треков.

---

### Ячейка 3. Сопоставление

**Вход:** `history_uris`, `playlist_uris`.
**Выход:** `in_both`, `only_hist` (сироты), `only_pl`.

**Логика:**
- `in_both = history ∩ playlists` — слушалось и лежит в плейлистах.
- `only_hist = history − playlists` — **сироты**.
- `only_pl = playlists − history` — в плейлистах, но не слушалось.

**Печатает:** три группы + проверку сходимости.

---

### Ячейка 4. Сохранение списков сирот

**Вход:** `only_hist`.
**Выход:** файлы в `data/Exportify/orphans_to_add/`.

**Логика:**
- Чистит `only_hist` от `None` и пустых.
- Сортирует URI.
- Сохраняет `orphans_uris.csv` — все сироты одной колонкой.
- Разбивает на батчи по 10 000 (лимит Spotify на плейлист).
- Сохраняет `orphans_batch_NN.csv`.

**Печатает:** счётчики по каждому файлу, всего сирот, число батчей.

---

### Ячейка 5. Развилка `METHOD`

**Выход:** переменная `METHOD`.

**Значения:**
- `'api'` — автоматическое создание плейлистов через Spotify API.
- `'manual'` — печатается инструкция для ручной загрузки.

---

### Ячейка 6a. Создание плейлистов через API

**Условие:** `METHOD == 'api'`.
**Вход:** `SPOTIFY`, `orphans_sorted`, `n_batches`.

**Логика:**
- OAuth через `spotipy`.
- Собирает все существующие плейлисты (с пагинацией) — защита от дублей.
- Для каждого батча:
  - если `__ORPHANS_NN` уже есть и число треков совпадает — **пропускает**;
  - если есть, но число другое — **конфликт**, не трогает;
  - если нет — создаёт и добавляет треки батчами по 100 (лимит API).
- Сохраняет `created_playlists.json` со статусами `created` / `skipped`.

**Выход:** плейлисты в Spotify + `created_playlists.json`.

---

### Ячейка 6b. Ручная инструкция

**Условие:** `METHOD == 'manual'`.

**Что делает:** печатает пошаговую инструкцию:
- **Шаг 1.** Создать приватные плейлисты `__ORPHANS_NN` в **десктопном** Spotify (веб не поддерживает массовую вставку).
- **Шаг 2.** Скопировать всю колонку `spotify_track_uri` из CSV → вставить в плейлист через Ctrl+V. Spotify принимает весь батч за одну вставку.
- **Шаг 3.** Экспортировать каждый плейлист через Exportify.
- **Шаг 4.** Положить CSV в `data/Exportify/exportify_orphan_export/`.
- **Шаг 5.** Запустить `00_1_build_full_history.ipynb`.

**Ничего не создаёт, только текст.**

---

### Ячейка 7. Финальная проверка

**Что проверяет:**
- Сироты из этого прогона (`orphans_uris.csv`).
- Что уже выгружено в `exportify_orphan_export/`.
- Покрытие: сколько сирот уже с фичами, сколько осталось.
- Аудио-фичи в CSV экспорта: наличие колонки `Valence`, доля треков с фичами.

**Печатает:**
- Сколько сирот найдено.
- Сколько выгружено.
- Процент покрытия.
- Процент треков с фичами (valence не NaN).

---

### Итог

После запуска ноута:

- `data/Exportify/orphans_to_add/orphans_uris.csv` — все сироты
- `data/Exportify/orphans_to_add/orphans_batch_NN.csv` — батчи по 10 000
- `data/Exportify/orphans_to_add/created_playlists.json` — метаданные плейлистов (при `METHOD='api'`)

Дальше: Exportify → CSV с фичами → `data/Exportify/exportify_orphan_export/` → запуск `00_1`.

## `00_0_orphans_detect.ipynb` — что решает

Ноутбук решает **одну конкретную проблему**: в истории прослушиваний есть треки, которые слушались, но ни в один плейлист пользователя не попали. Такие треки называются **сиротами**. Без них пайплайн работает неполно — Exportify отдаёт аудио-фичи (`valence`, `energy`, `danceability` и другие) только для треков, лежащих в плейлистах. Сироты остаются без фич, а значит не попадают в Sound-кластеры, не отображаются в жанровом анализе, не срабатывают в большинстве индексов дашборда. На реальных данных таких треков оказалось 18 395 из 25 629 — **72% от всей истории**. Игнорировать их нельзя.

## Как решает

Логика в три шага. Первое — читает все JSON с историей прослушиваний из privacy export, собирает уникальные `spotify_track_uri`. Второе — читает все CSV из Exportify (плейлисты + лайки), тоже собирает уникальные URI. Третье — вычитает: `сироты = история − плейлисты`. Результат сохраняется в CSV-файл `orphans_uris.csv` и разбивается на батчи по 10 000 треков (лимит Spotify на один плейлист).

Дальше **развилка**. Exportify не умеет «дать фичи по списку URI» — он умеет только экспортировать плейлисты. Значит сирот надо сначала **положить в плейлист**, чтобы Exportify их увидел. Два пути: через Spotify API (ноутбук сам создаёт приватные плейлисты `__ORPHANS_01`, `__ORPHANS_02` и добавляет туда треки батчами по 100 — всё за минуту) или вручную (пользователь создаёт плейлисты в десктопном Spotify и вставляет URI через Ctrl+V — десктоп принимает весь батч за одну вставку). После этого — Exportify экспортирует плейлисты, пользователь кладёт CSV в `data/Exportify/exportify_orphan_export/`.

## Что даёт

На выходе — три состояния, которые ноутбук показывает в финальной ячейке:
1. **Сколько сирот найдено** — 18 395 на реальных данных.
2. **Сколько из них уже выгружено** через Exportify — процент покрытия.
3. **Сколько выгруженных треков имеют аудио-фичи** — проверка, что Exportify не вернул пустые колонки.

Если всё чисто — следующий ноутбук (`00_1_build_full_history.ipynb`) подхватывает CSV и обогащает историю фичами. Если чего-то не хватает — ноутбук прямо говорит, на каком шаге остановился и что делать.

Ключевая идея: сироты — **единственный ручной шаг** во всём пайплайне. Всё остальное автоматизировано. Ноутбук делает его максимально безболезненным: находит сирот сам, создаёт плейлисты сам (если API), проверяет результат сам. Пользователю остаётся один клик в Exportify.